In [ ]:
#%% md
# Ноутбук 1: Непрерывная новизна события + гипотеза "stale news" + regression по волатильности/объёму
#
# В этом ноутбуке совмещены три взаимосвязанные гипотезы про ОДНУ и ту же ось —
# "насколько новость нова и как рынок на неё реагирует":
#
#   Гипотеза A (Event Novelty / Similarity Gap) — вместо бинарного дубль/не дубль считаем
#       непрерывную метрику новизны: число дней с последней похожей новости по тому же тикеру.
#   Гипотеза B (Stale News, Tetlock 2011) — рынок слабее реагирует на "устаревшие" новости
#       (маленький similarity gap), и после них должен быть разворот доходности.
#   Гипотеза C (богатые рыночные метрики) — вместо |доходности| используем волатильность
#       (диапазон high-low) и объём торгов как более чувствительные к новостям метрики,
#       и разделяем поток на "свежий"/"устаревший" вместо единого счётчика новостей.
#
# Специально НЕ смешиваем сюда идеи про выбор эмбеддинг-модели, LLM-валидацию кластеров
# или потоковую обработку — это отдельные ноутбуки 02-05, чтобы не путать причины эффектов.
#
# Входные данные: тот же telegram_news_raw.csv, что и в базовом ноутбуке дедупликации.
#%%


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('novelty_stale_news_results')
OUT_DIR.mkdir(exist_ok=True)
WINDOW_DAYS = 14          # то же окно, что и в базовом ноутбуке
STALE_GAP_DAYS = 7        # порог "устарелости": similarity gap < 7 дней = новость считается stale
SIM_THRESHOLD = 0.5       # порог сходства для similarity gap (мягче порога дублей — нам нужна "похожесть", а не точный дубль)

df = pd.read_csv(DATA_PATH, parse_dates=['post_dttm']).reset_index(drop=True)

def normalize_text(text):
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df['text_normalized'] = df['data'].fillna('').map(normalize_text)
df = df[df.text_normalized.str.len().gt(0)].copy().reset_index(drop=True)
print(f'Новостей после базовой очистки: {len(df):,}')

In [ ]:
def load_moex_tickers():
    url = 'https://iss.moex.com/iss/engines/stock/markets/shares/securities.json'
    try:
        resp = requests.get(url, params={'iss.meta': 'off'}, timeout=20)
        resp.raise_for_status()
        payload = resp.json()['securities']
        idx = payload['columns'].index('SECID')
        tickers = sorted({row[idx] for row in payload['data'] if row[idx]})
        if tickers:
            return tickers
        raise ValueError('пустой список')
    except Exception as exc:
        print(f'MOEX ISS недоступен ({exc}), использую запасной список тикеров.')
        return ['SBER', 'GAZP', 'LKOH', 'GMKN', 'NVTK', 'ROSN', 'TATN', 'MGNT',
                'MTSS', 'YDEX', 'VTBR', 'CHMF', 'ALRS', 'PLZL', 'MOEX', 'AFLT',
                'PHOR', 'RUAL', 'POLY', 'SNGS']

MOEX_TICKERS = load_moex_tickers()
ticker_pattern = re.compile(r'(?:\$)?\b(' + '|'.join(re.escape(t) for t in MOEX_TICKERS) + r')\b')
df['tickers_found'] = df['data'].map(lambda t: sorted(set(ticker_pattern.findall(str(t or '')))))
df_ticker = df[df.tickers_found.map(len) > 0].explode('tickers_found').rename(columns={'tickers_found': 'ticker'})
print(f'Новостей с тикером: {df.tickers_found.map(len).gt(0).sum():,} из {len(df):,}')
print(f'Строк после explode по тикерам: {len(df_ticker):,}')

In [ ]:
#%% md
### Гипотеза A: Event Similarity Gap (непрерывная новизна)
# Что проверяем: вместо бинарного "дубль/не дубль" даём каждой новости непрерывную оценку
#     новизны — число дней с последней ПОХОЖЕЙ новости (cosine similarity >= SIM_THRESHOLD)
#     ПО ТОМУ ЖЕ ТИКЕРУ за последние 100 дней (аналог RavenPack Event Similarity Gap).
# Как применяем: считаем TF-IDF по всему корпусу один раз, затем для каждого тикера берём
#     его новости, отсортированные по времени, и для каждой находим ближайшую по времени
#     похожую новость того же тикера в скользящем окне; gap = число дней до неё
#     (или MAX_LOOKBACK_DAYS, если такой не было — новость "полностью новая").
# Как оцениваем результат: смотрим распределение gap (гистограмма), долю "полностью новых"
#     новостей, и переиспользуем gap как фичу в регрессии в конце ноутбука (гипотеза C).
# Примечание по сложности: двойной цикл ниже — O(n^2) в рамках ОДНОГО тикера в пределах
#     временного окна; для датасета из ~19 тыс. новостей и умеренного числа упоминаний на
#     тикер это приемлемо, но на больших объёмах стоит заменить на top-k поиск (как в
#     базовом ноутбуке) вместо полного перебора.
#%%
MAX_LOOKBACK_DAYS = 100

tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=2, max_features=200_000)
tfidf_vectors = tfidf.fit_transform(df.text_normalized)
df['_row_id'] = np.arange(len(df))

def similarity_gap_for_ticker(rows, vectors):
    """rows: подтаблица df_ticker для одного тикера, произвольного порядка."""
    rows = rows.sort_values('post_dttm')
    idxs = rows['_row_id'].to_numpy()
    dates = rows['post_dttm'].to_numpy()
    sub = vectors[idxs]
    sims = (sub @ sub.T).toarray()
    gaps = np.full(len(rows), MAX_LOOKBACK_DAYS, dtype=float)
    for i in range(len(rows)):
        window_start = dates[i] - np.timedelta64(MAX_LOOKBACK_DAYS, 'D')
        best_gap = MAX_LOOKBACK_DAYS
        for j in range(i - 1, -1, -1):
            if dates[j] < window_start:
                break
            if sims[i, j] >= SIM_THRESHOLD:
                gap_days = (dates[i] - dates[j]) / np.timedelta64(1, 'D')
                best_gap = min(best_gap, gap_days)
        gaps[i] = best_gap
    return pd.Series(gaps, index=rows.index)

gap_parts = []
for ticker, rows in tqdm(df_ticker.groupby('ticker'), desc='Similarity gap по тикерам'):
    gap_parts.append(similarity_gap_for_ticker(rows, tfidf_vectors))
df_ticker['similarity_gap_days'] = pd.concat(gap_parts).reindex(df_ticker.index)
df_ticker['is_stale'] = df_ticker['similarity_gap_days'] < STALE_GAP_DAYS

# --- Оценка гипотезы A ---
fig, ax = plt.subplots(figsize=(10, 4))
df_ticker['similarity_gap_days'].clip(upper=MAX_LOOKBACK_DAYS).hist(bins=40, ax=ax)
ax.set_title('Распределение Event Similarity Gap (дни с последней похожей новости)')
ax.set_xlabel('Similarity gap, дней')
ax.set_ylabel('Число новостей')
plt.tight_layout()
plt.savefig(OUT_DIR / 'similarity_gap_hist.png', dpi=150)
plt.show()

share_fully_novel = (df_ticker['similarity_gap_days'] >= MAX_LOOKBACK_DAYS).mean()
print(f'Доля "полностью новых" новостей (gap = {MAX_LOOKBACK_DAYS}): {share_fully_novel:.1%}')
print(f'Доля "устаревших" (gap < {STALE_GAP_DAYS} дн.): {df_ticker.is_stale.mean():.1%}')
df_ticker[['ticker', 'post_dttm', 'similarity_gap_days', 'is_stale']].to_csv(
    OUT_DIR / 'similarity_gap.csv', index=False, encoding='utf-8-sig')

In [ ]:
#%% md
### Гипотеза B: Stale News (Tetlock, 2011)
# Что проверяем: рынок слабее реагирует на "устаревшую" новость (маленький similarity gap),
#     и после устаревшей новости чаще случается разворот доходности в следующие дни.
# Как применяем: берём top-N тикеров по числу упоминаний, тянем их дневные котировки с MOEX
#     ISS, считаем доходность в день новости (day 0) и через 1/5 торговых дней. Делим новости
#     на "stale" (gap < STALE_GAP_DAYS) и "fresh" (остальные), сравниваем средние |доходности|
#     и последующий разворот между группами.
# Как оцениваем результат: таблица средних |return| на день 0 и знаковой доходности на
#     день+1/+5 по группам stale/fresh + t-тест на разницу средних |return|. Ожидание по
#     Tetlock: у stale новостей меньше первоначальная реакция и заметнее разворот.
#%%
TOP_N_TICKERS = 15

def load_moex_security_history(secid, board='TQBR', start=None, end=None):
    base = f'https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/{board}/securities/{secid}.json'
    frames, start_pos = [], 0
    while True:
        params = {'iss.meta': 'off', 'start': start_pos}
        if start: params['from'] = start
        if end: params['till'] = end
        try:
            resp = requests.get(base, params=params, timeout=20)
            resp.raise_for_status()
        except Exception:
            return pd.DataFrame()
        payload = resp.json()['history']
        rows = payload['data']
        if not rows:
            break
        frames.append(pd.DataFrame(rows, columns=payload['columns']))
        start_pos += len(rows)
        if len(rows) < 100:
            break
    if not frames:
        return pd.DataFrame()
    hist = pd.concat(frames, ignore_index=True)
    hist['TRADEDATE'] = pd.to_datetime(hist['TRADEDATE'])
    return hist[['TRADEDATE', 'CLOSE']].dropna().drop_duplicates('TRADEDATE').sort_values('TRADEDATE')

top_tickers = df_ticker['ticker'].value_counts().head(TOP_N_TICKERS).index.tolist()
print('Топ тикеров для анализа:', top_tickers)

market_start = df.post_dttm.min().date().isoformat()
market_end = (df.post_dttm.max() + pd.Timedelta(days=10)).date().isoformat()

prices = {}
for t in tqdm(top_tickers, desc='Загрузка котировок MOEX'):
    hist = load_moex_security_history(t, start=market_start, end=market_end)
    if not hist.empty:
        s = hist.set_index('TRADEDATE')['CLOSE'].pct_change()
        prices[t] = s

print(f'Получены котировки для {len(prices)} из {len(top_tickers)} тикеров.')

In [ ]:
def nearest_trading_return(returns, event_date, horizon_days=0):
    """Доходность на horizon_days торговых дней вперёд от ближайшей даты >= event_date."""
    idx = returns.index.searchsorted(pd.Timestamp(event_date))
    pos = idx + horizon_days
    if 0 <= pos < len(returns):
        return returns.iloc[pos]
    return np.nan

records = []
ticker_events = df_ticker[df_ticker.ticker.isin(prices.keys())]
for _, row in tqdm(ticker_events.iterrows(), total=len(ticker_events), desc='Событийные доходности'):
    ret = prices[row.ticker]
    day0 = nearest_trading_return(ret, row.post_dttm.normalize(), 0)
    day1 = nearest_trading_return(ret, row.post_dttm.normalize(), 1)
    day5 = nearest_trading_return(ret, row.post_dttm.normalize(), 5)
    records.append({
        'ticker': row.ticker, 'post_dttm': row.post_dttm,
        'similarity_gap_days': row.similarity_gap_days, 'is_stale': row.is_stale,
        'ret_day0': day0, 'ret_day1': day1, 'ret_day5': day5
    })
events = pd.DataFrame(records).dropna(subset=['ret_day0'])
print(f'Событий с доступной доходностью: {len(events):,}')

# --- Оценка гипотезы B ---
from scipy import stats

abs_day0 = events.groupby('is_stale')['ret_day0'].apply(lambda s: s.abs().mean())
signed_day1 = events.groupby('is_stale')['ret_day1'].mean()
signed_day5 = events.groupby('is_stale')['ret_day5'].mean()

report_stale = pd.DataFrame({
    '|return| в день новости': abs_day0,
    'return день+1 (разворот)': signed_day1,
    'return день+5 (разворот)': signed_day5,
}).rename(index={True: f'stale (gap < {STALE_GAP_DAYS} дн.)', False: 'fresh'})
print(report_stale)

fresh_abs = events.loc[~events.is_stale, 'ret_day0'].abs().dropna()
stale_abs = events.loc[events.is_stale, 'ret_day0'].abs().dropna()
if len(fresh_abs) > 1 and len(stale_abs) > 1:
    t_stat, p_val = stats.ttest_ind(stale_abs, fresh_abs, equal_var=False)
    print(f'\nT-тест |return day0| stale vs fresh: t={t_stat:.3f}, p={p_val:.4f}')
    print('Гипотеза Tetlock подтверждается, если |return| у stale ЗАМЕТНО МЕНЬШЕ и p < 0.05.')
else:
    print('Недостаточно наблюдений для t-теста в одной из групп.')

report_stale.to_csv(OUT_DIR / 'stale_vs_fresh_returns.csv', encoding='utf-8-sig')

In [ ]:
#%% md
### Гипотеза C: волатильность/объём вместо |доходности|, с учётом новизны
# Что проверяем: улучшится ли объясняющая сила регрессии, если (a) использовать более
#     чувствительные к новостям рыночные метрики — внутридневной диапазон (high-low)/close
#     и объём торгов, вместо простого |return|, и (b) разделить новостной поток на "свежий"
#     и "устаревший" (по similarity gap), а не считать все новости одинаково важными.
# Как применяем: берём дневную историю IMOEX (HIGH/LOW/CLOSE/объём), считаем дневные
#     счётчики fresh/stale новостей по всему рынку, строим OLS для трёх целевых переменных
#     (|return|, диапазон, объём) на два регрессора (fresh_count, stale_count).
# Как оцениваем результат: сравниваем R² и знаки/значимость коэффициентов у fresh vs stale —
#     ожидание: fresh_count должен иметь больший и более значимый коэффициент, чем
#     stale_count, если гипотеза "устаревшие новости несут меньше информации" верна.
#%%
try:
    import statsmodels.api as sm
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    import statsmodels.api as sm

def load_moex_index_ohlcv(secid='IMOEX', board='SNDX', start=None, end=None):
    base = f'https://iss.moex.com/iss/history/engines/stock/markets/index/boards/{board}/securities/{secid}.json'
    frames, start_pos = [], 0
    while True:
        params = {'iss.meta': 'off', 'start': start_pos}
        if start: params['from'] = start
        if end: params['till'] = end
        resp = requests.get(base, params=params, timeout=20)
        resp.raise_for_status()
        payload = resp.json()['history']
        rows = payload['data']
        if not rows:
            break
        frames.append(pd.DataFrame(rows, columns=payload['columns']))
        start_pos += len(rows)
        if len(rows) < 100:
            break
    if not frames:
        return pd.DataFrame()
    hist = pd.concat(frames, ignore_index=True)
    hist['TRADEDATE'] = pd.to_datetime(hist['TRADEDATE'])
    return hist

try:
    imoex = load_moex_index_ohlcv('IMOEX', 'SNDX', market_start, market_end)
except Exception as exc:
    print(f'Не удалось получить OHLCV IMOEX: {exc}')
    imoex = pd.DataFrame()

if imoex.empty or not {'HIGH', 'LOW', 'CLOSE'}.issubset(imoex.columns):
    print('Регрессия по волатильности/объёму пропущена: нет нужных полей (HIGH/LOW) в ответе MOEX ISS.')
else:
    imoex = imoex.sort_values('TRADEDATE').reset_index(drop=True)
    imoex['return'] = imoex['CLOSE'].pct_change()
    imoex['abs_return'] = imoex['return'].abs()
    imoex['range_pct'] = (imoex['HIGH'] - imoex['LOW']) / imoex['CLOSE']
    vol_col = 'VOLRUR' if 'VOLRUR' in imoex.columns else ('VALUE' if 'VALUE' in imoex.columns else None)
    imoex['day'] = imoex['TRADEDATE'].dt.date

    daily_gap = df_ticker.assign(day=df_ticker.post_dttm.dt.date)
    fresh_counts = daily_gap[~daily_gap.is_stale].groupby('day').size()
    stale_counts = daily_gap[daily_gap.is_stale].groupby('day').size()
    news_daily = pd.DataFrame({'fresh_count': fresh_counts, 'stale_count': stale_counts}).fillna(0)

    targets = ['abs_return', 'range_pct'] + ([vol_col] if vol_col else [])
    reg_df = imoex.set_index('day')[targets].join(news_daily, how='inner').dropna()

    if len(reg_df) < 5:
        print(f'Слишком мало пересекающихся торговых дней ({len(reg_df)}) для регрессии.')
    else:
        for target in targets:
            X = sm.add_constant(reg_df[['fresh_count', 'stale_count']])
            y = reg_df[target]
            model = sm.OLS(y, X).fit()
            print(f'--- Цель: {target} (n={int(model.nobs)}) ---')
            print(model.summary().tables[1])
            print(f'R^2 = {model.rsquared:.4f}\n')
        reg_df.to_csv(OUT_DIR / 'fresh_vs_stale_regression.csv', encoding='utf-8-sig')

In [ ]:
# Итог ноутбука 1:
#  - similarity_gap.csv — непрерывная новизна каждой новости по тикеру (гипотеза A)
#  - stale_vs_fresh_returns.csv — сравнение реакции рынка stale/fresh (гипотеза B)
#  - fresh_vs_stale_regression.csv — регрессия волатильности/объёма на fresh/stale поток (гипотеза C)
print('Готово. Файлы результатов сохранены в', OUT_DIR.resolve())